# P11: attention'li neyro-tarjimon (Seq2Seq)
**Mavzu:** LSTM enkoder-dekoder, Bahdanau attention, tarjima, BLEU, attention heatmap
**Kun:** 12-kun amaliyoti — 3-iyul 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L11 — Seq2Seq va Attention (`d11_seq2seq_attention.pdf`)
**Kapstone modul:** m11 `Seq2SeqTranslator` (pedagogik)
**Ma'lumot:** OPUS-100 uz-en (~20k). Offline: `d12_checkpoints/uz_en_mini.txt` (original parallel).

## Bugungi maqsadlar (ma'ruza [C] dan)
1. Attention softmax ni qo'lda tekshiramiz ($\alpha=\mathrm{softmax}([2,1,3])=[0.245,0.090,0.665]$).
2. LSTM enkoder-dekoder + Bahdanau attention tarjimon quramiz.
3. Sodda gaplarni tarjima qilamiz; attention heatmap chizamiz.
4. BLEU bilan baholaymiz (kichik data → demo-sifatli BLEU, halol).

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | seedlar, OFFLINE_FALLBACK, HAS_TORCH, HAS_NLTK |
| §2 Yaxlit natija | 8 daq | tayyor translate + BLEU demo |
| §3 Periferiya (PRIMM) | 17 daq | uz-en parallel korpus + tokenizatsiya; BLEU |
| §4 Yadro | 35 daq | attention softmax (qulflangan), seq2seq+attention, translate, BLEU, heatmap |
| §5 Loyihaga ulash | 13 daq | m11 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | SOV->SVO; attention heatmap nimani ko'rsatadi |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq, `torch` va `nltk` bayroqlari. GPU shart emas (CPU).

In [ ]:
# Kaggle: torch + nltk oldindan o'rnatilgan (GPU bo'lsa tezlashadi, talab emas)
import os, sys, random
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
    print("torch mavjud:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch yo'q — lug'at-asosli tarjimon (offline).")

try:
    import nltk
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False
    print("nltk yo'q — toza-python BLEU ishlatiladi.")

MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")
sys.path.insert(0, MODULES)
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Tayyor `Seq2SeqTranslator` parallel korpusda o'qitiladi va o'zbekcha gaplarni inglizchaga
tarjima qiladi. (Kichik korpus → BLEU demo-sifatli, lekin tizim ishlaydi.)

In [ ]:
from m11_seq2seq_translator import Seq2SeqTranslator

PARALLEL = os.path.join("..", "..", "course", "practices", "d12_checkpoints", "uz_en_mini.txt")
if not os.path.exists(PARALLEL):
    PARALLEL = os.path.join("course", "practices", "d12_checkpoints", "uz_en_mini.txt")

def load_parallel(path):
    """uz<TAB>en juftliklar -> (src_texts, tgt_texts)."""
    src, tgt = [], []
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if not line.strip():
            continue
        u, e = line.split("\t")
        src.append(u); tgt.append(e)
    return src, tgt

src, tgt = load_parallel(PARALLEL)
print("Juftlar:", len(src))

mt_demo = Seq2SeqTranslator(); mt_demo.train(src, tgt, epochs=30)
print("Tarjima:", "men kitob o'qidim", "->", mt_demo.translate("men kitob o'qidim"))


## 3. Tayyor kod bloki — parallel korpus va BLEU (PRIMM)
Bu bo'lim periferiya. Tarjima ma'lumoti — manba(uz)$\to$maqsad(en) juftliklar. BLEU
tarjimani etalonga taqqoslaydi (nltk bo'lsa undan, aks holda toza-python).

**Bashorat qiling:** "men kitob o'qidim" / "i read a book" juftida nechta uz va nechta en token bor?

In [ ]:
# PERIFERIYA — to'liq beriladi. tokenizatsiya va BLEU.
def tokenize(s):
    return s.split()

print("Manba tokenlar:", tokenize("men kitob o'qidim"))
print("Maqsad tokenlar:", tokenize("i read a book"))

# BLEU — nltk bor bo'lsa undan, aks holda m11.bleu (toza-python)
def bleu_corpus(refs, hyps):
    if HAS_NLTK:
        from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
        sm = SmoothingFunction().method1
        return corpus_bleu([[tokenize(r)] for r in refs],
                           [tokenize(h) for h in hyps], smoothing_function=sm)
    # toza-python fallback (m11.bleu)
    from m11_seq2seq_translator import Seq2SeqTranslator
    return Seq2SeqTranslator().bleu(refs, hyps)

print("Sinov BLEU:", round(bleu_corpus(["i read a book"], ["i read a book"]), 3))


**Tekshiring:**
1. BLEU bir xil jumlalar uchun 1.0 ga yaqinmi? Nega?
2. Nega manba va maqsad uchun \textbf{alohida} lug'at kerak?
3. Teacher forcing nima — o'qitishda dekoderga to'g'ri so'z beriladimi yoki bashorat?

**O'zgartiring:** o'z uz-en juftingizni qo'shib, tokenlanishini ko'ring.

In [ ]:
# PERIFERIYA — teacher forcing sxemasi (torch yo'lida; izohli).
if HAS_TORCH:
    print("Teacher forcing: o'qitishda dekoderga OLDINGI to'g'ri (gold) so'z beriladi,")
    print("bashorat emas — bu o'qishni tezlashtiradi va barqarorlashtiradi.")
else:
    print("Offline: lug'at-asosli tarjimon (so'z-tekislash) — teacher forcing shart emas.")


### Checkpoint A
Orqada qolsangiz — parallel korpusni qaytadan yuklaydi.

In [ ]:
if OFFLINE_FALLBACK or "src" not in dir():
    src, tgt = load_parallel(PARALLEL)
print("Checkpoint: korpus tayyor —", len(src), "juft")


## 4. Asosiy mavzu — tarjimon (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning attention misolini takrorlaymiz.

### 4A. Namuna (men qilaman): attention softmax
Ma'ruza L11 [I3]: 3 enkoder holati energiyalari $e=[2,1,3]$. Attention vaznlari ---
softmax. Toza-numpy (torch shart emas).

In [ ]:
# Attention softmax (toza-numpy) — L11 [I3]
e = np.array([2.0, 1.0, 3.0])          # 3 enkoder holati energiyalari
alpha = np.exp(e) / np.exp(e).sum()
print("alpha =", np.round(alpha, 3))

assert np.allclose(alpha, [0.245, 0.090, 0.665], atol=1e-3)   # Ma'ruza L11 [I3]-slayd
print("To'g'ri! attention alpha = [0.245, 0.090, 0.665]")
# DIQQAT: bu alignment (e'tibor) vaznlari — L9 temperature emas (bir xil matematik, boshqa ma'no)


### 4B. Birgalikda (biz qilamiz): Seq2Seq + attention tarjimonni o'qitamiz
`Seq2SeqTranslator` ichida LSTM enkoder + Bahdanau attention + LSTM dekoder (Kaggle) yoki
lug'at-asosli tarjimon (offline). CPU uchun kichik o'lcham va kam epoch.

In [ ]:
mt = Seq2SeqTranslator()
# === SIZNING KODINGIZ (taxminan 2 qator) ===
# 1) mt ni src/tgt ustida o'qiting: epochs=30
# 2) "men kitob o'qidim" ni translate qiling
mt.train(...)
out = ...
print(out)


In [ ]:
assert len(mt._src2i) > 0 and len(mt._tgt2i) > 0, "Lug'at bo'sh — train() ni tekshiring."
assert isinstance(out, str), "translate() str qaytarishi kerak."
print("Ajoyib! Tarjimon o'qidi — manba lug'at:", len(mt._src2i), "maqsad lug'at:", len(mt._tgt2i))


### 4C. Birgalikda (biz qilamiz): tarjima va BLEU
Bir nechta gapni tarjima qilib, BLEU bilan baholaymiz. Kichik data → BLEU demo-sifatli.

In [ ]:
# === SIZNING KODINGIZ (taxminan 3 qator) ===
# 1) src[:5] ni tarjima qiling (hyps)
# 2) refs = tgt[:5]
# 3) mt.bleu(refs, hyps) ni hisoblang
hyps = ...
refs = ...
score = ...
print("BLEU:", round(score, 3))


In [ ]:
assert isinstance(hyps, list) and all(isinstance(h, str) for h in hyps), "Tarjimalar str ro'yxati."
assert isinstance(score, float) and 0.0 <= score <= 1.0, "BLEU float [0,1] oralig'ida bo'lishi kerak."
print("To'g'ri! Tarjima + BLEU ishladi (BLEU demo-sifatli — kichik data, halol).")


### 4D. Mustaqil (siz qilasiz): attention heatmap
Attention vaznlari matritsasini (dekoder qadami × enkoder pozitsiyasi) chizing — qaysi
manba so'ziga qaralganini ko'rsatadi (alignment). Tayanch yo'q.

In [ ]:
# === SIZNING KODINGIZ (taxminan 10 qator) ===
# 1) src/tgt tokenlar uchun attention matritsa A (maqsad x manba) tuzing
# 2) har qatorni softmax-simon normallang (yig'indi 1)
# 3) matplotlib (Agg) bilan imshow heatmap chizing (manba/maqsad teglar bilan)
# 4) assert: A.shape == (len(tgt), len(src)); har qator yig'indisi 1
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
...


## 5. Loyihaga ulash — m11 `Seq2SeqTranslator`
Bugungi ish `capstone/modules/m11_seq2seq_translator.py` da (shartnoma `capstone/contracts.py`).
Pedagogik (yakuniy pipelineda emas), torch-ixtiyoriy + nltk-ixtiyoriy BLEU; save/load YO'Q.

In [ ]:
from m11_seq2seq_translator import Seq2SeqTranslator

m = Seq2SeqTranslator()
for metod in ["train", "translate", "bleu"]:
    assert callable(getattr(m, metod, None)), f"m11.{metod} mavjud emas!"
m.train(src[:10], tgt[:10], epochs=3)
assert isinstance(m.translate("men non yedim"), str), "translate() str qaytaradi."
assert 0.0 <= m.bleu(["i ate bread"], ["i ate bread"]) <= 1.0, "bleu() float [0,1]."
print("m11 Seq2SeqTranslator shartnomaga mos: train / translate / bleu (pedagogik)")


```bash
git add capstone/modules/m11_seq2seq_translator.py
git commit -m "P11: m11 Seq2SeqTranslator — attention NMT"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba:** "men kitob o'qidim" (SOV) inglizchada "i read a book" (SVO) bo'ladi —
so'z tartibi o'zgaradi. Modelingiz buni qanday tarjima qildi?

In [ ]:
for s in ["men kitob o'qidim", "u maktabga bordi", "biz olma yedik"]:
    print(f"{s} -> {mt.translate(s)}")
print("Eslatma: attention so'z tartibidan mustaqil bog'lanish (alignment) quradi (SOV<->SVO).")


**Savol (o'ylab ko'ring):** attention heatmap nima ko'rsatadi (qaysi manba so'ziga qaralgani)?
O'zbekcha SOV gapni inglizcha SVO ga tarjima qilganda attention so'z tartibini qanday
"qayta tartiblaydi"? Kichik data nega BLEU ni past qiladi?

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_